# A* Search

In [ ]:
!pip install -q pyrosm networkx folium scikit-learn

In [ ]:
import os
import requests

url = "https://download.geofabrik.de/asia/bangladesh-latest.osm.pbf"
file_path = "/content/bangladesh.osm.pbf"

if not os.path.exists(file_path):
    response = requests.get(url, stream=True, timeout=120)
    response.raise_for_status()

    with open(file_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)

print("Bangladesh Dataset Downloaded Successfully")

Bangladesh Dataset Downloaded Successfully


In [ ]:
from pyrosm import OSM
import networkx as nx

osm = OSM(file_path)

G = osm.get_graph(
    network_type="driving",
    graph_type="networkx"
)

print("Road Network Created Successfully")
print("Total Nodes:", G.number_of_nodes())
print("Total Edges:", G.number_of_edges())

Building the graph of bangladesh.osm.pbf took 06:12


Road Network Created Successfully
Total Nodes: 643202
Total Edges: 1542061


In [ ]:
import math
import numpy as np
from sklearn.neighbors import BallTree

start_location = (23.7330, 90.3920)
goal_location = (23.8433, 90.3978)

node_ids = list(G.nodes())

coordinates = np.radians([
    [G.nodes[n]["y"], G.nodes[n]["x"]]
    for n in node_ids
])

tree = BallTree(coordinates, metric="haversine")

def nearest_node(location):
    index = tree.query(
        np.radians([location]), k=1
    )[1][0][0]

    return node_ids[index]

start = nearest_node(start_location)
goal = nearest_node(goal_location)

def heuristic(a, b):
    lat1, lon1 = G.nodes[a]["y"], G.nodes[a]["x"]
    lat2, lon2 = G.nodes[b]["y"], G.nodes[b]["x"]

    lat1, lon1, lat2, lon2 = map(
        math.radians, [lat1, lon1, lat2, lon2]
    )

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    h = math.sin(dlat / 2)**2
    h += math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2)**2

    return 6371000 * 2 * math.asin(math.sqrt(h))

print("Start Node:", start)
print("Goal Node:", goal)

Start Node: 4369503575
Goal Node: 4901057540


In [ ]:
import time

start_time = time.perf_counter()

path = nx.astar_path(
    G,
    start,
    goal,
    heuristic=heuristic,
    weight="length"
)

execution_time = time.perf_counter() - start_time

def get_edge(u, v):
    edges = G.get_edge_data(u, v)

    return min(
        edges.values(),
        key=lambda x: x["length"]
    )

total_distance = sum(
    get_edge(u, v)["length"]
    for u, v in zip(path[:-1], path[1:])
)

print("========== A* SEARCH RESULTS ==========")
print("Search Status: Path Found")
print("Total Nodes:", len(path))
print("Total Distance:", round(total_distance, 2), "meters")
print("Total Distance:", round(total_distance / 1000, 3), "KM")
print("Execution Time:", round(execution_time, 4), "seconds")

========== A* SEARCH RESULTS ==========
Search Status: Path Found
Total Nodes: 127
Total Distance: 14296.99 meters
Total Distance: 14.297 KM
Execution Time: 0.0871 seconds


In [ ]:
import folium

m = folium.Map(
    location=start_location,
    zoom_start=12,
    tiles=None,
    control_scale=True
)

folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
    attr="Tiles © Esri",
    name="Esri Street Map",
    overlay=False,
    control=True
).add_to(m)

route_coordinates = []

for u, v in zip(path[:-1], path[1:]):
    edge = get_edge(u, v)

    if "geometry" in edge:
        route_coordinates.extend(
            [(lat, lon) for lon, lat in edge["geometry"].coords]
        )
    else:
        route_coordinates.extend([
            (G.nodes[u]["y"], G.nodes[u]["x"]),
            (G.nodes[v]["y"], G.nodes[v]["x"])
        ])

folium.Marker(
    start_location,
    popup="START: Dhaka University",
    tooltip="Starting Point",
    icon=folium.Icon(color="green", icon="play")
).add_to(m)

folium.Marker(
    goal_location,
    popup="DESTINATION: Hazrat Shahjalal International Airport",
    tooltip="Destination",
    icon=folium.Icon(color="red", icon="flag")
).add_to(m)

folium.PolyLine(
    route_coordinates,
    color="blue",
    weight=6,
    opacity=0.9,
    tooltip="A* Optimal Route"
).add_to(m)

folium.map.Marker(
    [(start_location[0] + goal_location[0]) / 2,
     (start_location[1] + goal_location[1]) / 2],
    icon=folium.DivIcon(
        html=f"""
        <div style="background:white;padding:8px;border-radius:6px;
        border:1px solid #555;font-size:12px;width:180px;">
        <b>A* Search</b><br>
        Distance: {total_distance/1000:.2f} KM<br>
        Time: {execution_time:.4f} sec
        </div>
        """
    )
).add_to(m)

m.fit_bounds(route_coordinates, padding=(40, 40))

folium.LayerControl().add_to(m)

m

# Geedy BFS

In [ ]:
!pip install -q pyrosm networkx folium scikit-learn

In [ ]:
import os
import requests

url = "https://download.geofabrik.de/asia/bangladesh-latest.osm.pbf"
file_path = "/content/bangladesh.osm.pbf"

if not os.path.exists(file_path):
    response = requests.get(url, stream=True, timeout=120)
    response.raise_for_status()

    with open(file_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)

print("Bangladesh Dataset Downloaded Successfully")

Bangladesh Dataset Downloaded Successfully


In [ ]:
from pyrosm import OSM
import networkx as nx

osm = OSM(file_path)

G = osm.get_graph(
    network_type="driving",
    graph_type="networkx"
)

print("Road Network Created Successfully")
print("Total Nodes:", G.number_of_nodes())
print("Total Edges:", G.number_of_edges())

Building the graph of bangladesh.osm.pbf took 06:10


Road Network Created Successfully
Total Nodes: 643202
Total Edges: 1542061


In [ ]:
import math
import numpy as np
import heapq
from sklearn.neighbors import BallTree

start_location = (23.7330, 90.3920)
goal_location = (23.8433, 90.3978)

node_ids = list(G.nodes())

coordinates = np.radians([
    [G.nodes[n]["y"], G.nodes[n]["x"]]
    for n in node_ids
])

tree = BallTree(coordinates, metric="haversine")

def nearest_node(location):
    index = tree.query(
        np.radians([location]), k=1
    )[1][0][0]

    return node_ids[index]

start = nearest_node(start_location)
goal = nearest_node(goal_location)

def heuristic(a, b):
    lat1, lon1 = G.nodes[a]["y"], G.nodes[a]["x"]
    lat2, lon2 = G.nodes[b]["y"], G.nodes[b]["x"]

    lat1, lon1, lat2, lon2 = map(
        math.radians, [lat1, lon1, lat2, lon2]
    )

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    h = math.sin(dlat / 2)**2
    h += math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2)**2

    return 6371000 * 2 * math.asin(math.sqrt(h))

print("Start Node:", start)
print("Goal Node:", goal)

Start Node: 4369503575
Goal Node: 4901057540


In [ ]:
def greedy_best_first_search(G, start, goal):

    counter = 0
    priority_queue = [
        (heuristic(start, goal), counter, start)
    ]

    visited = set()
    parent = {start: None}

    while priority_queue:

        _, _, current = heapq.heappop(priority_queue)

        if current in visited:
            continue

        visited.add(current)

        if current == goal:
            path = []
            node = goal

            while node is not None:
                path.append(node)
                node = parent[node]

            return path[::-1]

        for neighbor in G.neighbors(current):

            if neighbor not in visited and neighbor not in parent:

                parent[neighbor] = current
                counter += 1

                heapq.heappush(
                    priority_queue,
                    (heuristic(neighbor, goal), counter, neighbor)
                )

    return None


import time

start_time = time.perf_counter()

path = greedy_best_first_search(G, start, goal)

execution_time = time.perf_counter() - start_time

def get_edge(u, v):
    edges = G.get_edge_data(u, v)

    return min(
        edges.values(),
        key=lambda x: x["length"]
    )

if path:
    total_distance = sum(
        get_edge(u, v)["length"]
        for u, v in zip(path[:-1], path[1:])
    )

    print("========== GREEDY SEARCH RESULTS ==========")
    print("Search Status: Path Found")
    print("Total Nodes:", len(path))
    print("Total Distance:", round(total_distance, 2), "meters")
    print("Total Distance:", round(total_distance / 1000, 3), "KM")
    print("Execution Time:", round(execution_time, 4), "seconds")

else:
    print("No Route Found")

========== GREEDY SEARCH RESULTS ==========
Search Status: Path Found
Total Nodes: 196
Total Distance: 21196.94 meters
Total Distance: 21.197 KM
Execution Time: 0.0142 seconds


In [ ]:

import folium
from IPython.display import display
from shapely.geometry import LineString

if path and len(path) > 1:

    m = folium.Map(
        location=start_location,
        zoom_start=12,
        tiles=None,
        control_scale=True
    )

    folium.TileLayer(
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
        attr="Tiles © Esri",
        name="Esri Street Map",
        overlay=False,
        control=True
    ).add_to(m)

    route_coordinates = []
    total_distance = 0

    for u, v in zip(path[:-1], path[1:]):

        edge_data = G.get_edge_data(u, v)

        if edge_data is None:
            continue

        if G.is_multigraph():
            edge = min(
                edge_data.values(),
                key=lambda x: x.get("length", float("inf"))
            )
        else:
            edge = edge_data

        total_distance += edge.get("length", 0)

        if "geometry" in edge:
            coords = list(edge["geometry"].coords)
            route_coordinates.extend(
                [(lat, lon) for lon, lat in coords]
            )
        else:
            route_coordinates.extend([
                (G.nodes[u]["y"], G.nodes[u]["x"]),
                (G.nodes[v]["y"], G.nodes[v]["x"])
            ])

    if route_coordinates:

        folium.Marker(
            start_location,
            popup="START: Dhaka University",
            tooltip="Starting Point",
            icon=folium.Icon(color="green", icon="play")
        ).add_to(m)

        folium.Marker(
            goal_location,
            popup="DESTINATION: Hazrat Shahjalal International Airport",
            tooltip="Destination",
            icon=folium.Icon(color="red", icon="flag")
        ).add_to(m)

        folium.PolyLine(
            route_coordinates,
            color="red",
            weight=6,
            opacity=0.9,
            tooltip="Greedy Best-First Search Route"
        ).add_to(m)

        folium.Marker(
            [
                (start_location[0] + goal_location[0]) / 2,
                (start_location[1] + goal_location[1]) / 2
            ],
            icon=folium.DivIcon(
                html=f"""
                <div style="
                    background:white;
                    padding:8px;
                    border-radius:6px;
                    border:1px solid #555;
                    font-size:12px;
                    width:190px;">
                    <b>Greedy Best-First Search</b><br>
                    Distance: {total_distance / 1000:.2f} KM<br>
                    Time: {execution_time:.4f} sec
                </div>
                """
            )
        ).add_to(m)

        m.fit_bounds(route_coordinates, padding=(40, 40))

        folium.LayerControl().add_to(m)

        display(m)

    else:
        print("Route coordinates could not be generated.")

else:
    print("No route available for visualization.")
